In [2]:
import json

with open('/content/attribute_train.data','r') as f:
  train_attributes = [json.loads(line) for line in f.readlines()]

with open('/content/attribute_val.data','r') as f:
  val_attributes = [json.loads(line) for line in f.readlines()]

with open('/content/attribute_train.solution','r') as f:
  train_solutions = [json.loads(line) for line in f.readlines()]

with open('/content/attribute_val.solution','r') as f:
  val_solutions = [json.loads(line) for line in f.readlines()]


In [3]:
train_attributes[0]

{'indoml_id': 0,
 'title': 'Enclume Angled Pot Hook, Set of 6, Use with Pot Racks, Copper Plated',
 'store': 'Enclume',
 'details_Manufacturer': 'Enclume'}

In [4]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is i

In [10]:
from datasets import Dataset

# creating text input and output dataset for T5 model

def build_dataset(attributes, solutions):

  attributes_list = []
  solutions_list = []

  solution_map = {
      sol['indoml_id']: sol for sol in solutions
  }    # creating a map as alignment of attibutes and solution are not ensured

  for item in attributes:
    id = item['indoml_id']
    sol = solution_map[id]

    if not sol:
      continue    # no target found

    input_text = (
        f"Classify product:\n"
        f"Title: {item.get('title', '')}\n"
        f"Store: {item.get('store', '')}\n"
        f"Manufacturer: {item.get('details_Manufacturer', '')}"
    )

    # Convert solution dict to JSON string
    output_text = json.dumps({
        "details_Brand": sol.get("details_Brand", ""),
        "L0_category": sol.get("L0_category", ""),
        "L1_category": sol.get("L1_category", ""),
        "L2_category": sol.get("L2_category", ""),
        "L3_category": sol.get("L3_category", ""),
        "L4_category": sol.get("L4_category", "na")
    })

    attributes_list.append(input_text)
    solutions_list.append(output_text)

  dataset = Dataset.from_dict({
      'input': attributes_list,
      'target': solutions_list
  })

  return dataset

In [11]:
train_dataset = build_dataset(train_attributes, train_solutions)

In [12]:
train_dataset

Dataset({
    features: ['input', 'target'],
    num_rows: 443499
})

In [13]:
validation_dataset = build_dataset(val_attributes, val_solutions)

validation_dataset

Dataset({
    features: ['input', 'target'],
    num_rows: 95035
})

In [14]:
from datasets import DatasetDict

dataset = DatasetDict({
    'train': train_dataset,
    'validation': validation_dataset
})

In [15]:
from huggingface_hub import notebook_login, whoami

notebook_login()

In [16]:
whoami()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'type': 'user',
 'id': '66a797b589b3e71262932d0d',
 'name': 'SurAyush',
 'fullname': 'Ayush Sur',
 'email': 'ayushsur26@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'periodEnd': None,
 'isPro': False,
 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/RZJZW_w0wdVoOmQY250lR.png',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Apr',
   'role': 'write',
   'createdAt': '2025-04-29T01:49:17.243Z'}}}

In [18]:
dataset.push_to_hub('ProductTitle-To-Category')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/444 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/96 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/SurAyush/ProductTitle-To-Category/commit/ac1758bbb42716e0e44ed99343d11090cbbee8c9', commit_message='Upload dataset', commit_description='', oid='ac1758bbb42716e0e44ed99343d11090cbbee8c9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/SurAyush/ProductTitle-To-Category', endpoint='https://huggingface.co', repo_type='dataset', repo_id='SurAyush/ProductTitle-To-Category'), pr_revision=None, pr_num=None)